
Before running: NRC Emotion Lexicon
- This notebook needs the NRC Emotion Lexicon which is free for research use.
- Download it from the official page: https://saifmohammad.com/WebPages/NRC-Emotion-Lexicon.htm
- Unzip it and find the file `NRC-Emotion-Lexicon-Wordlevel-v0.92.txt` and put that file in the `lexicons/` folder of this project.


In [1]:
import os

NRC_PATH = "lexicons/NRC-Emotion-Lexicon-Wordlevel-v0.92.txt"
if not os.path.exists(NRC_PATH):
    raise FileNotFoundError(
        "Please download the NRC Emotion Lexicon (free) from "
        "https://saifmohammad.com/WebPages/NRC-Emotion-Lexicon.htm "
        "and put NRC-Emotion-Lexicon-Wordlevel-v0.92.txt in the lexicons/ folder. "
        "It is not included because its terms of use do not allow redistribution."
    )
print("NRC lexicon found")

NRC lexicon found


In [2]:
import pandas as pd
import numpy as np
import re, io, pickle, urllib.request, os

reviews = pd.read_parquet("data/processed/reviews_clean.parquet")

data = reviews[reviews["has_crowd"] & reviews["long_enough"]].copy()
print(len(data))   

169384


In [3]:

URL = ("https://raw.githubusercontent.com/Watts-Lab/team_comm_tools/"
       "2b2067831f5a4b2e624f050d8d9ae1fde74cad4e/"
       "src/team_comm_tools/features/assets/certainty.pkl")

text = pickle.loads(urllib.request.urlopen(URL).read())  
certainty_lex = pd.read_csv(io.StringIO(text))           
print(certainty_lex.shape)   

(3485, 4)


In [4]:
certainty_lex = certainty_lex.sort_values(["NumWords", "NumCharacters"], ascending=False)
cert_score = dict(zip(certainty_lex["Word"], certainty_lex["Certainty"]))     # phrase -> score
cert_rank  = dict(zip(certainty_lex["Word"],
                      zip(certainty_lex["NumWords"], certainty_lex["NumCharacters"])))

cert_pattern = re.compile(r"(?=\b(" + "|".join(re.escape(w) for w in certainty_lex["Word"]) + r")\b)")

def certainty(text):
    """Average certainty (0-9) of the lexicon phrases in a review.
    Returns NaN (no score) if the review has none (Decision 21)."""
    if not isinstance(text, str):
        return np.nan
    text = text.lower().replace("’", "'")
    found = [(m.start(), m.start() + len(m.group(1)), m.group(1))
             for m in cert_pattern.finditer(text)]
    found.sort(key=lambda f: cert_rank[f[2]], reverse=True)
    kept, used = [], []
    for start, end, phrase in found:
        if all(end <= u_start or start >= u_end for u_start, u_end in used):
            kept.append(phrase)
            used.append((start, end))
    if not kept:
        return np.nan
    return np.mean([cert_score[p] for p in kept])

for s in ["This is definitely the worst cream ever.",
          "I think maybe it didn't suit my skin.",
          "I wasn't sure. I think that it is not enough information.",
          "Nice color."]:
    print(round(certainty(s), 2) if not np.isnan(certainty(s)) else "no score", "|", s)

8.59 | This is definitely the worst cream ever.
4.04 | I think maybe it didn't suit my skin.
2.69 | I wasn't sure. I think that it is not enough information.
no score | Nice color.


In [5]:
if os.path.exists("data/processed/feat_certainty.parquet"):
    print("feat_certainty.parquet already exists - loaded, not recomputed.")
    data = data.drop(columns=["certainty"], errors="ignore").merge(pd.read_parquet("data/processed/feat_certainty.parquet"), on=["parent_asin", "user_id"], how="left")
else:
    data["certainty"] = data["text_clean"].apply(certainty)

print("Reviews with a certainty score:", data["certainty"].notna().sum())
print("Reviews with NO certainty phrase (no score):", data["certainty"].isna().sum())
print(data["certainty"].describe())

feat_certainty.parquet already exists - loaded, not recomputed.
Reviews with a certainty score: 147880
Reviews with NO certainty phrase (no score): 21504
count    147880.000000
mean          6.216842
std           0.952450
min           0.590000
25%           5.730000
50%           6.323333
75%           6.846667
max           8.810000
Name: certainty, dtype: float64


In [6]:
if os.path.exists("data/processed/feat_certainty.parquet"):
    print("feat_certainty.parquet already exists.")
else:
    data[["parent_asin", "user_id", "certainty"]].to_parquet("data/processed/feat_certainty.parquet", index=False)
    print("Saved:", len(data), "rows")

feat_certainty.parquet already exists.


In [7]:
%pip install empath


[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
from empath import Empath
empath = Empath()

EMPATH_CATS = ["positive_emotion", "negative_emotion", "anger", "sadness", "money"]

for s in ["I hate this, it was a waste of money and I'm so angry.",
          "I love it, it makes me so happy and it smells wonderful.",
          "Sadly it broke my heart, I was so disappointed and upset.",
          "The bottle is blue."]:
    result = empath.analyze(s, categories=EMPATH_CATS, normalize=True)
    print({k: round(v * 100, 1) for k, v in result.items()}, "|", s)

{'positive_emotion': 0.0, 'negative_emotion': 7.7, 'anger': 0.0, 'sadness': 0.0, 'money': 7.7} | I hate this, it was a waste of money and I'm so angry.
{'positive_emotion': 16.7, 'negative_emotion': 0.0, 'anger': 0.0, 'sadness': 0.0, 'money': 0.0} | I love it, it makes me so happy and it smells wonderful.
{'positive_emotion': 0.0, 'negative_emotion': 9.1, 'anger': 0.0, 'sadness': 0.0, 'money': 0.0} | Sadly it broke my heart, I was so disappointed and upset.
{'positive_emotion': 0.0, 'negative_emotion': 0.0, 'anger': 0.0, 'sadness': 0.0, 'money': 0.0} | The bottle is blue.


In [9]:
import glob

nrc_file = glob.glob("lexicons/NRC-Emotion-Lexicon-Wordlevel-v0.92.txt", recursive=True)[0]

nrc = pd.read_csv(nrc_file, sep="\t", names=["word", "emotion", "flag"], keep_default_na=False)
print(nrc_file, nrc.shape, "unique words:", nrc["word"].nunique())

nrc = nrc[nrc["flag"] == 1]      

NRC_CATS = ["positive", "negative", "anger", "sadness"]
nrc_words = {cat: set(nrc.loc[nrc["emotion"] == cat, "word"]) for cat in NRC_CATS}
for cat in NRC_CATS:
    print(cat, len(nrc_words[cat]), "words")

from empath import Empath
money_words = set(Empath().cats["money"])
print("money", len(money_words), "words")

lexicons/NRC-Emotion-Lexicon-Wordlevel-v0.92.txt (141540, 3) unique words: 14154
positive 2308 words
negative 3316 words
anger 1245 words
sadness 1187 words
money 107 words


In [10]:
def get_words(text):
    """Lowercase and split into words; punctuation removed ("angry." -> "angry")."""
    if not isinstance(text, str):
        return []
    return re.findall(r"[a-z]+(?:'[a-z]+)?", text.lower().replace("’", "'"))

def emotion_features(text):
    """Percentage of the review's words that are in each list."""
    words = get_words(text)
    n = len(words)
    if n == 0:
        return pd.Series({"pos_emotion_pct": np.nan, "neg_emotion_pct": np.nan,
                          "anger_pct": np.nan, "sadness_pct": np.nan, "money_pct": np.nan})
    return pd.Series({
        "pos_emotion_pct": sum(w in nrc_words["positive"] for w in words) / n * 100,
        "neg_emotion_pct": sum(w in nrc_words["negative"] for w in words) / n * 100,
        "anger_pct":       sum(w in nrc_words["anger"]    for w in words) / n * 100,
        "sadness_pct":     sum(w in nrc_words["sadness"]  for w in words) / n * 100,
        "money_pct":       sum(w in money_words           for w in words) / n * 100,
    })

test = pd.Series(["I hate this, it was a waste of money and I'm so angry.",
                  "I love it, it makes me so happy and it smells wonderful.",
                  "Sadly it broke my heart, I was so disappointed and upset.",
                  "The bottle is blue."])
print(test.apply(emotion_features).round(1).assign(text=test.str[:35]).to_string())

   pos_emotion_pct  neg_emotion_pct  anger_pct  sadness_pct  money_pct                                 text
0              7.7             23.1       23.1          7.7        7.7  I hate this, it was a waste of mone
1             25.0              0.0        0.0          0.0        0.0  I love it, it makes me so happy and
2              0.0             36.4       18.2         36.4        0.0  Sadly it broke my heart, I was so d
3              0.0              0.0        0.0         25.0        0.0                  The bottle is blue.


In [11]:
from collections import Counter

word_counts = Counter()
for t in data["text_clean"]:
    word_counts.update(get_words(t))

lists = {**nrc_words, "money": money_words}
for cat, wordset in lists.items():
    top = [(w, c) for w, c in word_counts.most_common() if w in wordset][:20]
    print(f"\n{cat.upper()}:")
    print(", ".join(f"{w} ({c:,})" for w, c in top))


POSITIVE:
love (37,493), good (31,956), recommend (15,742), perfect (10,711), pretty (10,424), clean (9,462), money (8,828), happy (8,150), worth (7,410), found (7,131), polish (7,109), top (6,921), received (6,692), cream (6,231), job (5,662), cute (4,894), full (4,846), beautiful (4,600), daughter (4,131), feeling (4,096)

NEGATIVE:
smell (14,086), small (9,795), wear (6,912), bad (6,450), disappointed (5,546), leave (5,090), cheap (4,327), problem (4,264), feeling (4,096), case (4,089), remove (4,069), waste (3,796), battery (2,850), broke (2,540), pain (2,536), broken (2,279), fits (2,227), bottom (2,198), scrub (2,135), fall (2,116)

ANGER:
smell (14,086), money (8,828), bad (6,450), disappointed (5,546), feeling (4,096), remove (4,069), hot (3,814), battery (2,850), broken (2,279), fits (2,227), lash (2,029), horrible (1,707), terrible (1,624), honest (1,538), hurt (1,305), hate (1,277), irritation (1,240), damage (1,186), complaint (1,181), crazy (1,159)

SADNESS:
bad (6,450), 

In [12]:
rows = []
for cat, wordset in lists.items():
    top = [(w, c) for w, c in word_counts.most_common() if w in wordset][:30]
    for rank, (w, c) in enumerate(top, start=1):
        rows.append({"list": cat, "rank": rank, "word": w, "count": c})

top_words = pd.DataFrame(rows)
print(top_words.pivot(index="rank", columns="list", values="word").to_string())

list         anger       money      negative    positive       sadness
rank                                                                  
1            smell       price         smell        love           bad
2            money         buy         small        good  disappointed
3              bad       money          wear   recommend         leave
4     disappointed    purchase           bad     perfect       problem
5          feeling       worth  disappointed      pretty       feeling
6           remove      buying         leave       clean          case
7              hot   expensive         cheap       money        remove
8          battery      seller       problem       happy          dark
9           broken      change       feeling       worth         broke
10            fits         pay          case       found          pain
11            lash        cost        remove      polish        broken
12        horrible        paid         waste         top        bottom
13    

In [13]:
for cat, wordset in lists.items():
    counts = {w: c for w, c in word_counts.items() if w in wordset}
    total = sum(counts.values())
    top10 = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:10]
    print(f"\n{cat.upper()} (all matched words: {total:,})")
    for w, c in top10:
        print(f"  {w:12s} {c / total * 100:5.1f}%")


POSITIVE (all matched words: 475,808)
  love           7.9%
  good           6.7%
  recommend      3.3%
  perfect        2.3%
  pretty         2.2%
  clean          2.0%
  money          1.9%
  happy          1.7%
  worth          1.6%
  found          1.5%

NEGATIVE (all matched words: 245,062)
  smell          5.7%
  small          4.0%
  wear           2.8%
  bad            2.6%
  disappointed   2.3%
  leave          2.1%
  cheap          1.8%
  problem        1.7%
  feeling        1.7%
  case           1.7%

ANGER (all matched words: 105,961)
  smell         13.3%
  money          8.3%
  bad            6.1%
  disappointed   5.2%
  feeling        3.9%
  remove         3.8%
  hot            3.6%
  battery        2.7%
  broken         2.2%
  fits           2.1%

SADNESS (all matched words: 114,954)
  bad            5.6%
  disappointed   4.8%
  leave          4.4%
  problem        3.7%
  feeling        3.6%
  case           3.6%
  remove         3.5%
  dark           3.1%
  broke     

In [14]:
doc_freq = Counter()
for t in data["text_clean"]:
    doc_freq.update(set(get_words(t)))       

min_reviews = 0.05 * len(data)             
print("A word must appear in at least", int(min_reviews), "reviews")

words_5pct = set()
for cat, wordset in lists.items():
    selected = [w for w in wordset if doc_freq[w] >= min_reviews]
    words_5pct.update(selected)
    covered = sum(word_counts[w] for w in selected)
    total = sum(word_counts[w] for w in wordset)
    print(f"{cat}: {len(selected)} words, covering {covered / total * 100:.0f}% of all matched words")

print("\nDifferent words to check:", len(words_5pct))
print(sorted(words_5pct))

A word must appear in at least 8469 reviews
positive: 5 words, covering 22% of all matched words
negative: 2 words, covering 10% of all matched words
anger: 1 words, covering 13% of all matched words
sadness: 0 words, covering 0% of all matched words
money: 2 words, covering 37% of all matched words

Different words to check: 9
['buy', 'good', 'love', 'perfect', 'pretty', 'price', 'recommend', 'small', 'smell']


In [15]:
import os

LIST_ORDER = ["anger", "sadness", "negative", "positive", "money"]
Y_MEANS = {  
    "positive": "the word itself shows liking or praise",
    "negative": "the word itself shows something bad or unpleasant",
    "anger":    "the word itself shows annoyance or anger",
    "sadness":  "the word itself shows sadness, disappointment or loss",
    "money":    "the word itself is about money, price, cost or value",
}

if os.path.exists("ai_labels/word_check.csv"):
    print("word_check.csv already exists, not overwritten.")
else:
    text_lower = data["text_clean"].str.lower().str.replace("’", "'", regex=False)
    snippets = {}   

    def get_snippets(word):
        """10 random short snippets containing the word, with the word in [CAPITALS]."""
        pattern = re.compile(r"\b" + re.escape(word) + r"\b")
        with_word = text_lower[text_lower.str.contains(pattern)]
        sample = with_word.sample(n=min(10, len(with_word)), random_state=42)
        out = []
        for t in sample:
            m = pattern.search(t)
            out.append("..." + t[max(0, m.start() - 80): m.start()] + "[" + word.upper() + "]"
                       + t[m.end(): m.end() + 80] + "...")
        return out

    rows = []
    for cat in LIST_ORDER:

        ranked = [w for w, c in word_counts.most_common() if w in lists[cat]][:40]
        for rank, word in enumerate(ranked, start=1):
            if word not in snippets:
                snippets[word] = get_snippets(word)
            for i, snip in enumerate(snippets[word], start=1):
                rows.append({"list": cat, "rank": rank, "word": word, "sentence": i,
                             "Y_means": Y_MEANS[cat], "snippet": snip, "emotional": ""})
    check = pd.DataFrame(rows)
    check.to_csv("ai_labels/word_check.csv", index=False, encoding="utf-8-sig")
    print("Saved word_check.csv:", len(check), "rows")

word_check.csv already exists, not overwritten.


In [16]:
import pandas as pd, numpy as np, re, os
from collections import Counter
from empath import Empath

reviews = pd.read_parquet("data/processed/reviews_clean.parquet")
data = reviews[reviews["has_crowd"] & reviews["long_enough"]].copy()

nrc = pd.read_csv("lexicons/NRC-Emotion-Lexicon-Wordlevel-v0.92.txt", sep="\t",
                  names=["word", "emotion", "flag"], keep_default_na=False)
nrc = nrc[nrc["flag"] == 1]
lists = {cat: set(nrc.loc[nrc["emotion"] == cat, "word"]) for cat in ["positive", "negative", "anger", "sadness"]}
lists["money"] = set(Empath().cats["money"])

dec = pd.read_csv("ai_labels/word_decisions.csv")
removed = {cat: set(dec[(dec["list"] == cat) & dec["checked"] & (dec["decision"] == "remove")]["word"])
           for cat in lists}
clean = {cat: lists[cat] - removed[cat] for cat in lists}
if os.path.exists("ai_labels/removed_words.csv"):
    print("removed_words.csv already exists - not overwritten.")
else:
    pd.DataFrame([(c, w) for c in lists for w in sorted(removed[c])],
                 columns=["list", "word"]).to_csv("ai_labels/removed_words.csv", index=False)

def get_words(text):
    if not isinstance(text, str):
        return []
    return re.findall(r"[a-z]+(?:'[a-z]+)?", text.lower().replace("’", "'"))

words = data["text_clean"].apply(get_words)
n = words.str.len()
ORIG  = {"positive": "pos_emotion_pct", "negative": "neg_emotion_pct", "anger": "anger_pct",
         "sadness": "sadness_pct", "money": "money_pct"}
CLEAN = {c: col.replace("_pct", "_clean_pct") for c, col in ORIG.items()}
already = os.path.exists("data/processed/feat_emotion.parquet")
if already:
    print("feat_emotion.parquet already exists - loaded, not recomputed.")
    data = data.drop(columns=list(ORIG.values()) + list(CLEAN.values()), errors="ignore").merge(pd.read_parquet("data/processed/feat_emotion.parquet"), on=["parent_asin", "user_id"], how="left")
else:
    for cat in lists:  
        data[ORIG[cat]]  = words.apply(lambda ws: sum(w in lists[cat] for w in ws)) / n * 100
        data[CLEAN[cat]] = words.apply(lambda ws: sum(w in clean[cat] for w in ws)) / n * 100

counts = Counter(w for ws in words for w in ws)
for cat in lists:
    total = sum(counts[w] for w in lists[cat])
    checked = set(dec[(dec["list"] == cat) & dec["checked"]]["word"])
    print(f"{cat}: removed {len(removed[cat])} words | checked words = "
          f"{sum(counts[w] for w in checked) / total * 100:.0f}% of matches | removed = "
          f"{sum(counts[w] for w in removed[cat]) / total * 100:.0f}%")

cols = list(ORIG.values()) + list(CLEAN.values())
print(data[list(CLEAN.values())].describe().round(2))
print("\nShare of reviews with a score of 0:\n", (data[list(CLEAN.values())] == 0).mean().round(3))
if not already:
    data[["parent_asin", "user_id"] + cols].to_parquet("data/processed/feat_emotion.parquet", index=False)
    print("Saved feat_emotion.parquet:", len(data), "rows")

removed_words.csv already exists - not overwritten.
feat_emotion.parquet already exists - loaded, not recomputed.
positive: removed 15 words | checked words = 52% of matches | removed = 15%
negative: removed 8 words | checked words = 34% of matches | removed = 21%
anger: removed 30 words | checked words = 77% of matches | removed = 60%
sadness: removed 35 words | checked words = 69% of matches | removed = 60%
money: removed 5 words | checked words = 91% of matches | removed = 40%
       pos_emotion_clean_pct  neg_emotion_clean_pct  anger_clean_pct  \
count              169384.00              169384.00        169384.00   
mean                    3.99                   1.84             0.41   
std                     3.18                   2.22             1.01   
min                     0.00                   0.00             0.00   
25%                     1.89                   0.00             0.00   
50%                     3.49                   1.35             0.00   
75%        

In [17]:
import importlib.util
if importlib.util.find_spec("en_core_web_sm") is None:
    %pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl
else:
    print("en_core_web_sm already installed - not reinstalled.")

en_core_web_sm already installed - not reinstalled.


In [18]:
import spacy
nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])   # we only need words + grammar

NEG_WORDS = {"no", "never", "nothing", "none", "nobody", "nowhere", "neither", "nor"}
I_WORDS = {"i", "me", "my", "mine", "myself"}

def spacy_features(doc):
    """Negation and first-person singular, as % of words (punctuation not counted)."""
    words = [t for t in doc if not t.is_punct and not t.is_space]
    n = len(words)
    if n == 0:
        return np.nan, np.nan
    neg = sum(1 for t in words if t.dep_ == "neg" or t.lower_ in NEG_WORDS)
    i_count = sum(1 for t in words if t.lower_ in I_WORDS)
    return neg / n * 100, i_count / n * 100

tests = ["I don't like it, it never worked for me.",
         "This is not what I expected and nothing happened.",
         "Great product, works well.",
         "My skin loves it and I'm happy."]
rows = []
for s in tests:
    neg, i_pct = spacy_features(nlp(s))
    rows.append({"sentence": s, "negation_%": round(neg, 1), "I_words_%": round(i_pct, 1)})
print(pd.DataFrame(rows).to_string(index=False))

                                         sentence  negation_%  I_words_%
         I don't like it, it never worked for me.        20.0       20.0
This is not what I expected and nothing happened.        22.2       11.1
                       Great product, works well.         0.0        0.0
                  My skin loves it and I'm happy.         0.0       25.0


In [19]:
from tqdm.auto import tqdm  

already = os.path.exists("data/processed/feat_spacy.parquet")
if already:
    print("feat_spacy.parquet already exists - loaded, not recomputed.")
    data = data.drop(columns=["negation_pct", "i_pct"], errors="ignore").merge(pd.read_parquet("data/processed/feat_spacy.parquet"), on=["parent_asin", "user_id"], how="left")
else:
    texts = data["text_clean"].fillna("").tolist()

    results = []
    for doc in tqdm(nlp.pipe(texts, batch_size=500), total=len(texts)):
        results.append(spacy_features(doc))

    data["negation_pct"] = [r[0] for r in results]
    data["i_pct"] = [r[1] for r in results]

print(data[["negation_pct", "i_pct"]].describe().round(2))
print("\nShare of reviews with a score of 0:")
print((data[["negation_pct", "i_pct"]] == 0).mean().round(3))

if not already:
    data[["parent_asin", "user_id", "negation_pct", "i_pct"]].to_parquet("data/processed/feat_spacy.parquet", index=False)
    print("Saved feat_spacy.parquet:", len(data), "rows")

feat_spacy.parquet already exists - loaded, not recomputed.
       negation_pct      i_pct
count     169384.00  169384.00
mean           2.12       6.36
std            2.15       3.85
min            0.00       0.00
25%            0.00       3.50
50%            1.89       6.25
75%            3.28       8.89
max           18.52      42.34

Share of reviews with a score of 0:
negation_pct    0.338
i_pct           0.092
dtype: float64


In [20]:
reasons = [l.strip() for l in open("lexicons/lexicon_reasons.txt", encoding="utf-8") if l.strip()]
mentions = [l.strip() for l in open("lexicons/lexicon_mentions.txt", encoding="utf-8") if l.strip()]
print(len(reasons), "reason phrases |", len(mentions), "mention phrases")

def make_pattern(phrases):
    """Same as in 03_wordlists: longer phrases first, whole words only."""
    phrases = sorted(phrases, key=len, reverse=True)
    return re.compile(r"\b(" + "|".join(re.escape(p) for p in phrases) + r")\b")

reasons_re, mentions_re = make_pattern(reasons), make_pattern(mentions)
text = data["text_clean"].str.lower().str.replace("’", "'", regex=False)

data["reasons_count"] = text.str.count(reasons_re)
data["mentions_count"] = text.str.count(mentions_re)
data["reasons_pct"] = data["reasons_count"] / data["n_words"] * 100
data["mentions_pct"] = data["mentions_count"] / data["n_words"] * 100
data["mentions_any"] = data["mentions_count"] > 0

print("Reviews with a reason phrase:", (data["reasons_count"] > 0).sum())
print("Reviews with a mention phrase:", data["mentions_any"].sum())

if os.path.exists("data/processed/feat_lexicons.parquet"):
    print("feat_lexicons.parquet already exists - not overwritten.")
else:
    data[["parent_asin", "user_id", "reasons_count", "reasons_pct",
          "mentions_count", "mentions_pct", "mentions_any"]].to_parquet("data/processed/feat_lexicons.parquet", index=False)
    print("Saved feat_lexicons.parquet")

13 reason phrases | 34 mention phrases
Reviews with a reason phrase: 24935
Reviews with a mention phrase: 4209
feat_lexicons.parquet already exists - not overwritten.


In [21]:
KEY = ["parent_asin", "user_id"]

reviews = pd.read_parquet("data/processed/reviews_clean.parquet")
features = reviews[reviews["has_crowd"] & reviews["long_enough"]].copy()
assert not features.duplicated(KEY).any(), "Two reviews share the same ID!"
n_start = len(features)

for f in ["data/processed/feat_certainty.parquet", "data/processed/feat_emotion.parquet",
          "data/processed/feat_spacy.parquet", "data/processed/feat_lexicons.parquet"]:
    part = pd.read_parquet(f)
    assert not part.duplicated(KEY).any(), f"Duplicate IDs in {f}"
    features = features.merge(part, on=KEY, how="left", validate="one_to_one")
    assert len(features) == n_start, f"Row count changed after {f}!"
    print(f"Joined {f}: {part.shape[1] - 2} columns")

print("\nBlank values per feature:")
print(features.drop(columns=reviews.columns, errors="ignore").isna().sum())

if os.path.exists("data/processed/features.parquet"):
    print("features.parquet already exists - not overwritten.")
else:
    features.to_parquet("data/processed/features.parquet", index=False)
    print("\nSaved features.parquet:", features.shape)

Joined data/processed/feat_certainty.parquet: 1 columns
Joined data/processed/feat_emotion.parquet: 10 columns
Joined data/processed/feat_spacy.parquet: 2 columns
Joined data/processed/feat_lexicons.parquet: 5 columns

Blank values per feature:
certainty                21504
pos_emotion_pct              0
neg_emotion_pct              0
anger_pct                    0
sadness_pct                  0
money_pct                    0
pos_emotion_clean_pct        0
neg_emotion_clean_pct        0
anger_clean_pct              0
sadness_clean_pct            0
money_clean_pct              0
negation_pct                 0
i_pct                        0
reasons_count                0
reasons_pct                  0
mentions_count               0
mentions_pct                 0
mentions_any                 0
dtype: int64
features.parquet already exists - not overwritten.
